In [128]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

In [129]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

In [130]:
# Cada fila son 5 mins
HORITZO = {30:6, 60:12}  # minuts : files

In [131]:
df_559_train = pd.read_csv('../data/559/559_train.csv', sep=';', header=None, names=cols)
df_559_test = pd.read_csv('../data/559/559_test.csv', sep=';', header=None, names=cols)

In [132]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    prep = df.copy()

    prep['timestamp'] = pd.to_datetime(
        dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute)
    )
    
    prep.sort_values('timestamp', inplace=True)

    # Coma decimal a punt
    fix_col = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in fix_col:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem tipo
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(df, columns=['meal_type'], dummy_na=False, prefix='meal')

    # Drop columnas amb casi tot NaN o valor constant
    prep = prep.drop(columns=["second","finger_stick","meal"])

    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill', limit=12)  # només fins 1 h enrere

    return prep

In [133]:
df_559_train = preprocess(df_559_train)
df_559_test = preprocess(df_559_test)

display(df_559_train.tail())
display(df_559_test.head())
display(df_559_test.tail())

,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_0,meal_1,meal_2,meal_3,meal_4,meal_5
12078,2022,1,17,23,35,161.0,"0,83",0,3,0,0,0,0,0,58.0,"0,000213","92,3","89,6",0,94,True,False,False,False,False,False
12079,2022,1,17,23,40,164.0,"0,83",0,3,0,0,0,0,0,58.0,"0,000201","92,3","89,6",0,94,True,False,False,False,False,False
12080,2022,1,17,23,45,168.0,"0,83",0,3,0,0,0,0,0,58.0,"0,000198","92,3","89,6",0,94,True,False,False,False,False,False
12081,2022,1,17,23,50,172.0,"0,83",0,3,0,0,0,0,0,57.0,"0,000192","92,3","89,6",0,94,True,False,False,False,False,False
12082,2022,1,17,23,55,176.0,"0,83",0,3,0,0,0,0,0,58.0,"0,000188","92,3","89,6",0,94,True,False,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_0,meal_1,meal_2,meal_3,meal_4
0,2022,1,18,0,0,179.0,"0,83",0,3,0,0,0,0,0,57.0,"0,000183","92,3","89,6",0,94,True,False,False,False,False
1,2022,1,18,0,5,183.0,"0,83",0,3,0,0,0,0,0,57.0,"0,000182","92,3","89,6",0,94,True,False,False,False,False
2,2022,1,18,0,10,187.0,"0,83",0,3,0,0,0,0,0,57.0,"0,000177","92,3","89,6",0,94,True,False,False,False,False
3,2022,1,18,0,15,191.0,"0,83",0,3,0,0,0,0,0,56.0,"0,000169","92,3","89,6",0,94,True,False,False,False,False
4,2022,1,18,0,20,195.0,"0,83",0,3,0,0,0,0,0,56.0,"0,000166","92,3","89,6",0,94,True,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_0,meal_1,meal_2,meal_3,meal_4
2871,2022,1,27,23,15,185.0,"1,25",0,3,0,0,0,0,0,NaN,0,0,0,0,0,True,False,False,False,False
2872,2022,1,27,23,20,183.0,"1,25",0,3,0,0,0,0,0,NaN,0,0,0,0,0,True,False,False,False,False
2873,2022,1,27,23,25,182.0,"1,25",0,3,0,0,0,0,0,NaN,0,0,0,0,0,True,False,False,False,False
2874,2022,1,27,23,30,180.0,"1,25",0,3,0,0,0,0,0,NaN,0,0,0,0,0,True,False,False,False,False
2875,2022,1,27,23,35,177.0,"1,25",0,3,0,0,0,0,0,NaN,0,0,0,0,0,True,False,False,False,False


In [134]:
display(df_559_train.info())
print('Shape train: ', df_559_train.shape)
print('Shape test: ', df_559_test.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12083 entries, 0 to 12082
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    12083 non-null  int64  
 1   month                   12083 non-null  int64  
 2   day                     12083 non-null  int64  
 3   hour                    12083 non-null  int64  
 4   minute                  12083 non-null  int64  
 5   glucose_level           11226 non-null  float64
 6   basal                   12083 non-null  object 
 7   bolus                   12083 non-null  object 
 8   sleep                   12083 non-null  int64  
 9   work                    12083 non-null  int64  
 10  stressors               12083 non-null  int64  
 11  hypo_event              12083 non-null  int64  
 12  illness                 12083 non-null  int64  
 13  exercise                12083 non-null  int64  
 14  basis_heart_rate        11827 non-null

None

Shape train:  (12083, 26)
Shape test:  (2876, 25)


In [135]:
targets = {}
features = {}


for h, fila in HORITZO.items():
    df = df_559_train.copy()
    df[f"y_{h}"] = df["glucose_level"].shift(-fila)

    df.dropna(inplace=True) # Eliminem files sense target
    
    targets[h] = df[f"y_{h}"]
    features[h] = df.drop(columns=[f"y_{h}"])

print(features)
print(targets)

{30:        year  month  day  hour  minute  glucose_level basal bolus  sleep  work  \
142    2021     12    7    12      55          240.0   0,9     0      0     0   
143    2021     12    7    13       0          229.0   0,9     0      0     0   
144    2021     12    7    13       5          223.0   0,9     0      0     0   
145    2021     12    7    13      10          214.0   0,9     0      0     0   
146    2021     12    7    13      15          210.0   0,9     0      0     0   
...     ...    ...  ...   ...     ...            ...   ...   ...    ...   ...   
12072  2022      1   17    23       5          149.0  0,83     0      3     0   
12073  2022      1   17    23      10          150.0  0,83     0      3     0   
12074  2022      1   17    23      15          152.0  0,83     0      3     0   
12075  2022      1   17    23      20          155.0  0,83     0      3     0   
12076  2022      1   17    23      25          156.0  0,83     0      3     0   

       stressors  hypo

In [ ]:
def add_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for m, steps in HORITZO.items():
        df[f'y_{m}'] = df['glucose_level'].shift(-steps)
    return df



In [136]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

models = {}

mean_score = make_scorer(mean_absolute_error, greater_is_better=False)
rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 3, 5]
}

for h, steps in HORITZO.items():
    y = train_df[f'glucose_level'].shift(-steps).dropna()
    X = train_df.loc[y.index, feature_cols]


    gs = RandomizedSearchCV(
        rf, param_grid, 
        n_iter=10,
        scoring=mean_score,
        cv=3, 
        n_jobs=-1, 
        random_state=42
        )
    
    gs.fit(X, y)
    models[h] = gs.best_estimator_



NameError: name 'train_df' is not defined

In [ ]:
# PREDICCIONS EN EL TEST
# S'ignoren els primers 60 minuts (12 files)
test = df_559_test.iloc[12:].reset_index(drop=True)

for h, fila in HORITZO.items():
    # Treiem les ultimes files perque no tindran prediccio
    Xt = test.iloc[:-fila].copy()
    preds = models[h].predict(Xt)
    out = pd.DataFrame({"idx_original": Xt.index, f"pred_glucose_t+{h}": preds})

    out.to_csv(f"../data/predicted/pred559_{h}min.csv", index=False)

    display(out.head())

### Evaluació del model

In [ ]:
metrics = []

param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 3, 5]
}

for h, fila in HORITZO.items():
    # 1) y_true: glucosa real desplaçada -h
    y_true = test["glucose_level"].iloc[:-fila].reset_index(drop=True)
    
    # 2) y_pred: lee tu CSV de predicciones
    y_pred = (pd.read_csv(f"../data/predicted/pred559_{h}min.csv")[f"pred_glucose_t+{h}"].reset_index(drop=True))

    # 3) MAE
    mae = mean_absolute_error(y_true, y_pred)
    # 4) RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    metrics.append({
        "Pacient": 559,
        "Horitzo": f"{h} min",
        "RMSE": rmse,
        "MAE": mae
    })

result_df = pd.DataFrame(metrics)

result_df.loc["PROMIG"] = ["-", "-", result_df["RMSE"].mean(), result_df["MAE"].mean()]

print("\nResultats del pacient 559")
print(result_df)



Resultats del pacient 559
       Pacient Horitzo       RMSE        MAE
0          559  30 min  29.184779  22.158562
1          559  60 min  29.113176  22.093154
PROMIG       -       -  29.148978  22.125858
